# Laboratorio 4 — Análisis de Datos GeoEspaciales
## Cianobacteria en los lagos de Atitlán y Amatitlán con Sentinel-2

Universidad del Valle de Guatemala — CC3084 Data Science — Semestre II 2026

In [1]:
# !pip install openeo rasterio numpy pandas matplotlib

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import openeo

# Las divisiones de los índices generan 0/0 en píxeles sin dato; se manejan como NaN.
np.seterr(divide="ignore", invalid="ignore")

DATOS = Path("data")
(DATOS / "raw").mkdir(parents=True, exist_ok=True)
(DATOS / "resultados").mkdir(parents=True, exist_ok=True)

---
## 1. Conexión al API de Sentinel-2

Usamos el módulo `openeo` contra el backend del *Copernicus Data Space Ecosystem*.
`authenticate_oidc()` hay que abrir el navegador para iniciar sesión con nuestro usuario de Copernicus Browser y el token queda guardado para las siguientes ejecuciones.

In [3]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()
connection

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=CLPQ-QHYV 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

---
## 2. Áreas de interés y fechas

11 fechas por lago, elegidas por su baja nubosidad. 

In [4]:
lago_atitlan = {
    "west": -91.326256,
    "east": -91.07151,
    "south": 14.5948,
    "north": 14.750979,
}
lago_amatitlan = {
    "west": -90.638065,
    "east": -90.512924,
    "south": 14.412347,
    "north": 14.493799,
}

LAGOS = {"Atitlan": lago_atitlan, "Amatitlan": lago_amatitlan}

In [5]:
# Fechas oficiales del laboratorio con su nubosidad reportada.
ESCENAS = pd.DataFrame(
    [
        ("Atitlan",   "2025-01-18",  0.02, "Sentinel-2B"),
        ("Atitlan",   "2025-04-13",  0.54, "Sentinel-2C"),
        ("Atitlan",   "2025-05-13",  4.37, "Sentinel-2C"),
        ("Atitlan",   "2025-07-17",  3.57, "Sentinel-2A"),
        ("Atitlan",   "2025-11-21",  3.15, "Sentinel-2A"),
        ("Atitlan",   "2025-12-29",  3.17, "Sentinel-2C"),
        ("Atitlan",   "2026-02-12",  0.04, "Sentinel-2B"),
        ("Atitlan",   "2026-03-24",  3.17, "Sentinel-2B"),
        ("Atitlan",   "2026-04-13",  0.01, "Sentinel-2B"),
        ("Atitlan",   "2026-04-28",  4.96, "Sentinel-2C"),
        ("Atitlan",   "2026-07-22",  4.02, "Sentinel-2B"),
        ("Amatitlan", "2025-01-28",  0.06, "Sentinel-2B"),
        ("Amatitlan", "2025-04-15",  0.09, "Sentinel-2A"),
        ("Amatitlan", "2025-04-28",  1.03, "Sentinel-2B"),
        ("Amatitlan", "2025-11-24",  0.50, "Sentinel-2B"),
        ("Amatitlan", "2026-01-08",  0.77, "Sentinel-2C"),
        ("Amatitlan", "2026-02-02",  0.39, "Sentinel-2B"),
        ("Amatitlan", "2026-02-07",  0.02, "Sentinel-2C"),  # cobertura parcial (~57%)
        ("Amatitlan", "2026-03-29",  0.01, "Sentinel-2C"),
        ("Amatitlan", "2026-04-13",  0.09, "Sentinel-2B"),
        ("Amatitlan", "2026-04-28",  4.96, "Sentinel-2C"),
        ("Amatitlan", "2026-06-19", 13.00, "Sentinel-2A"),
    ],
    columns=["lago", "fecha", "nubosidad", "satelite"],
)

ESCENAS

,lago,fecha,nubosidad,satelite
0,Atitlan,2025-01-18,0.02,Sentinel-2B
1,Atitlan,2025-04-13,0.54,Sentinel-2C
2,Atitlan,2025-05-13,4.37,Sentinel-2C
3,Atitlan,2025-07-17,3.57,Sentinel-2A
4,Atitlan,2025-11-21,3.15,Sentinel-2A
5,Atitlan,2025-12-29,3.17,Sentinel-2C
6,Atitlan,2026-02-12,0.04,Sentinel-2B
7,Atitlan,2026-03-24,3.17,Sentinel-2B
8,Atitlan,2026-04-13,0.01,Sentinel-2B
9,Atitlan,2026-04-28,4.96,Sentinel-2C


---
## 3. Descarga de los datos raster

Solo descargamos las 9 bandas que necesitan los tres índices, remuestreadas a
**20 m** (la resolución nativa de B05, B07, B8A, B11 y B12). Así evitamos bajar
escenas completas.

| Índice | Bandas |
|---|---|
| NDVI | B04, B08 |
| NDWI | B03, B08 |
| Cianobacteria (NDCI + máscara de agua) | B02, B03, B04, B05, B07, B08, B8A, B11, B12 |

Usamos el producto **L2A** (reflectancia de superficie, ya corregida
atmosféricamente), que es el mismo que usa el cuaderno de ejemplo del curso.

In [6]:
BANDAS = ["B02", "B03", "B04", "B05", "B07", "B08", "B8A", "B11", "B12"]


def descargar_escena(lago, fecha):
    """Descarga las bandas de un lago en una fecha. Si el archivo ya existe, no repite la descarga."""
    ruta = DATOS / "raw" / f"{lago}_{fecha}.tif"
    if ruta.exists():
        return ruta

    # openEO pide un intervalo; usamos [fecha, fecha+1] para traer solo ese día.
    dia_siguiente = (pd.Timestamp(fecha) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    cubo = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=LAGOS[lago],
        temporal_extent=[fecha, dia_siguiente],
        bands=BANDAS,
    ).resample_spatial(resolution=20, projection=32615)  # UTM 15N, 20 m

    job = cubo.create_job(out_format="GTIFF", title=f"{lago}_{fecha}")
    job.start_and_wait()
    job.get_results().download_file(ruta)
    return ruta

In [ ]:
# Descarga de las 22 escenas. Puede tardar; si se interrumpe, basta con volver a
# correr la celda: las escenas ya descargadas se saltan.
for fila in ESCENAS.itertuples():
    try:
        descargar_escena(fila.lago, fila.fecha)
        print("OK    ", fila.lago, fila.fecha)
    except Exception as error:
        print("FALLO ", fila.lago, fila.fecha, "->", error)

0:00:00 Job 'j-260813222536486fa2d41ba8cba657d5': send 'start'
0:00:02 Job 'j-260813222536486fa2d41ba8cba657d5': created (progress 0%)
0:00:07 Job 'j-260813222536486fa2d41ba8cba657d5': queued (progress 0%)
0:00:14 Job 'j-260813222536486fa2d41ba8cba657d5': queued (progress 0%)
0:00:22 Job 'j-260813222536486fa2d41ba8cba657d5': queued (progress 0%)
0:00:32 Job 'j-260813222536486fa2d41ba8cba657d5': queued (progress 0%)
0:00:44 Job 'j-260813222536486fa2d41ba8cba657d5': running (progress N/A)
0:01:00 Job 'j-260813222536486fa2d41ba8cba657d5': running (progress N/A)
0:01:19 Job 'j-260813222536486fa2d41ba8cba657d5': running (progress N/A)
0:01:44 Job 'j-260813222536486fa2d41ba8cba657d5': finished (progress 100%)
OK     Atitlan 2025-01-18
0:00:00 Job 'j-2608132227294ccbafb1a1ef307920a6': send 'start'
0:00:02 Job 'j-2608132227294ccbafb1a1ef307920a6': queued (progress 0%)
0:00:07 Job 'j-2608132227294ccbafb1a1ef307920a6': queued (progress 0%)
0:00:14 Job 'j-2608132227294ccbafb1a1ef307920a6': queued

---
## 4. Cálculo de los índices

Replicamos en Python el script **Cyanobacteria Chlorophyll-a NDCI**
(CyanoLakes, Kravitz & Matthews) de <https://custom-scripts.sentinel-hub.com>.
El script tiene tres partes:

1. **Máscara de agua**: combina MNDWI, NDWI, AWEI, NDVI y descarta zonas urbanas
   y suelo desnudo. Solo dentro de esa máscara tiene sentido el índice.
2. **NDCI** = (B05 − B04) / (B05 + B04). El borde rojo (B05, 705 nm) responde a
   la clorofila-a, que es el pigmento de la cianobacteria.
3. **Clorofila-a** en mg/m³ mediante el polinomio calibrado del script:
   `chl = 826.57·NDCI³ − 176.43·NDCI² + 19·NDCI + 4.071`.

El script además marca con FAI (*Floating Algae Index*) la vegetación flotante;
esos píxeles se excluyen porque no son floración de cianobacteria.

In [ ]:
def leer_bandas(ruta):
    """Lee el GeoTIFF y devuelve {banda: reflectancia 0-1}."""
    with rasterio.open(ruta) as src:
        datos = src.read().astype("float32") / 10000  # de DN a reflectancia
        nombres = list(src.descriptions)
    if not all(nombres):
        nombres = BANDAS  # openEO respeta el orden pedido en `bands`
    return dict(zip(nombres, datos))


def mascara_agua(b):
    """Detección de cuerpos de agua del script de Sentinel Hub (Gartner)."""
    ndvi = (b["B08"] - b["B04"]) / (b["B08"] + b["B04"])
    mndwi = (b["B03"] - b["B11"]) / (b["B03"] + b["B11"])
    ndwi = (b["B03"] - b["B08"]) / (b["B03"] + b["B08"])
    ndwi_hojas = (b["B08"] - b["B11"]) / (b["B08"] + b["B11"])
    aweish = b["B02"] + 2.5 * b["B03"] - 1.5 * (b["B08"] + b["B11"]) - 0.25 * b["B12"]
    aweinsh = 4 * (b["B03"] - b["B11"]) - (0.25 * b["B08"] + 2.75 * b["B11"])
    dbsi = (b["B11"] - b["B03"]) / (b["B11"] + b["B03"]) - ndvi

    agua = (
        (mndwi > 0.42) | (ndwi > 0.4) | (aweinsh > 0.1879)
        | (aweish > 0.1112) | (ndvi < -0.2) | (ndwi_hojas > 1)
    )
    urbano_o_suelo = (aweinsh <= -0.03) | (dbsi > 0)  # filtro de falsos positivos
    return agua & ~urbano_o_suelo


def calcular_indices(lago, fecha):
    """Devuelve los índices de una escena, con la clorofila-a limitada al agua."""
    b = leer_bandas(DATOS / "raw" / f"{lago}_{fecha}.tif")
    agua = mascara_agua(b)

    ndvi = (b["B08"] - b["B04"]) / (b["B08"] + b["B04"])
    ndwi = (b["B03"] - b["B08"]) / (b["B03"] + b["B08"])

    # Índice de cianobacteria
    ndci = (b["B05"] - b["B04"]) / (b["B05"] + b["B04"])
    chla = 826.57 * ndci**3 - 176.43 * ndci**2 + 19 * ndci + 4.071

    # Vegetación flotante: alta reflectancia en el borde rojo que no es cianobacteria
    fai = b["B07"] - b["B04"] - (b["B8A"] - b["B04"]) * (783 - 665) / (865 - 665)

    valido = agua & (fai <= 0.08)
    return {
        "rgb": np.dstack([b["B04"], b["B03"], b["B02"]]),
        "agua": agua,
        "ndvi": np.where(agua, ndvi, np.nan),
        "ndwi": np.where(agua, ndwi, np.nan),
        "chla": np.where(valido, chla, np.nan),
    }

In [ ]:
# Comprobación rápida con una escena
prueba = calcular_indices("Amatitlan", "2026-03-29")
print("Píxeles de agua:", int(prueba["agua"].sum()))
print("Clorofila-a (mg/m3)  media: %.1f   máx: %.1f"
      % (np.nanmean(prueba["chla"]), np.nanmax(prueba["chla"])))

---
## 5. Visualización de los índices

Para cada lago mostramos color verdadero, NDVI, NDWI y el índice de cianobacteria
en una fecha de referencia.

**Cómo leer los mapas:**
- **Color verdadero**: la imagen tal como se vería a simple vista.
- **NDVI** (verde alto): vegetación. Sobre el agua es negativo, salvo donde hay
  biomasa algal flotando, que lo empuja hacia arriba.
- **NDWI** (azul alto): agua. Sirve para verificar que la máscara cubre el lago.
- **Clorofila-a**: el índice de cianobacteria. Escala de 0 a 50 mg/m³; el rojo
  marca las zonas de floración.

In [ ]:
def realce(banda):
    """Estira el contraste entre los percentiles 2 y 98 para la imagen en color."""
    p2, p98 = np.nanpercentile(banda, (2, 98))
    return np.clip((banda - p2) / (p98 - p2), 0, 1)


def mapa_escena(lago, fecha):
    ix = calcular_indices(lago, fecha)
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))

    axes[0, 0].imshow(realce(ix["rgb"]))
    axes[0, 0].set_title("Color verdadero")

    capas = [
        (axes[0, 1], ix["ndvi"], "NDVI (sobre agua)", "YlGn", -0.5, 0.5),
        (axes[1, 0], ix["ndwi"], "NDWI (sobre agua)", "BrBG", -1, 1),
        (axes[1, 1], ix["chla"], "Cianobacteria: clorofila-a (mg/m3)", "turbo", 0, 50),
    ]
    for ax, capa, titulo, cmap, vmin, vmax in capas:
        imagen = ax.imshow(capa, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(titulo)
        fig.colorbar(imagen, ax=ax, shrink=0.75)

    for ax in axes.ravel():
        ax.axis("off")
    fig.suptitle(f"Lago de {lago} — {fecha}", fontsize=15)
    fig.tight_layout()
    plt.show()

In [ ]:
mapa_escena("Atitlan", "2026-04-13")

In [ ]:
mapa_escena("Amatitlan", "2026-03-29")

---
## 6. Análisis temporal

Para cada lago y fecha resumimos la clorofila-a **solo sobre los píxeles de agua**:

- `chla_media` y `chla_mediana`: nivel típico del lago ese día.
- `chla_p90`: intensidad de las zonas más afectadas.
- `pct_floracion`: porcentaje de la superficie del lago por encima de 10 mg/m³,
  el umbral de alerta de la OMS para cianobacteria en aguas recreativas.

También guardamos NDVI y NDWI medios sobre agua, que se usarán en el análisis de
correlación de la entrega final.

In [ ]:
UMBRAL_FLORACION = 10  # mg/m3 de clorofila-a (nivel de alerta OMS)

resumen = []
for fila in ESCENAS.itertuples():
    if not (DATOS / "raw" / f"{fila.lago}_{fila.fecha}.tif").exists():
        continue

    ix = calcular_indices(fila.lago, fila.fecha)
    chla = ix["chla"][~np.isnan(ix["chla"])]  # solo píxeles de agua válidos
    resumen.append({
        "lago": fila.lago,
        "fecha": pd.Timestamp(fila.fecha),
        "chla_media": chla.mean(),
        "chla_mediana": np.median(chla),
        "chla_p90": np.percentile(chla, 90),
        "pct_floracion": 100 * (chla > UMBRAL_FLORACION).mean(),
        "ndvi_agua": np.nanmean(ix["ndvi"]),
        "ndwi_agua": np.nanmean(ix["ndwi"]),
        "px_agua": int(ix["agua"].sum()),
    })

TEMPORAL = pd.DataFrame(resumen).sort_values(["lago", "fecha"]).reset_index(drop=True)
TEMPORAL.to_csv(DATOS / "resultados" / "serie_temporal.csv", index=False)
TEMPORAL.round(2)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=False)

for ax, lago in zip(axes, ["Atitlan", "Amatitlan"]):
    g = TEMPORAL[TEMPORAL.lago == lago]
    ax.plot(g.fecha, g.chla_media, marker="o", color="seagreen", label="Media")
    ax.plot(g.fecha, g.chla_p90, marker="^", ls="--", color="darkorange", label="Percentil 90")
    ax.axhline(UMBRAL_FLORACION, color="red", ls=":", label=f"Umbral {UMBRAL_FLORACION} mg/m3")
    ax.set_title(f"Lago de {lago} — evolución de la clorofila-a")
    ax.set_ylabel("Clorofila-a (mg/m3)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(DATOS / "resultados" / "serie_temporal.png", dpi=120)
plt.show()

### 6.1 Picos de floración y fechas críticas

In [ ]:
picos = TEMPORAL.loc[TEMPORAL.groupby("lago")["chla_media"].idxmax()]
criticas = TEMPORAL[TEMPORAL.chla_media > UMBRAL_FLORACION]

for lago, g in TEMPORAL.groupby("lago"):
    pico = picos[picos.lago == lago].iloc[0]
    crit = criticas[criticas.lago == lago]
    print(f"--- {lago} ---")
    print(f"  Rango de la media : {g.chla_media.min():.1f} a {g.chla_media.max():.1f} mg/m3")
    print(f"  Pico              : {pico.fecha:%Y-%m-%d} con {pico.chla_media:.1f} mg/m3 "
          f"({pico.pct_floracion:.0f}% del lago sobre el umbral)")
    print(f"  Fechas criticas   : {', '.join(crit.fecha.dt.strftime('%Y-%m-%d')) or 'ninguna'}")
    print(f"  Media por mes     :")
    print(g.groupby(g.fecha.dt.month).chla_media.mean().round(1).to_string())
    print()

In [ ]:
# Comparación directa entre lagos
fig, ax = plt.subplots(figsize=(11, 4.5))
for lago, g in TEMPORAL.groupby("lago"):
    ax.plot(g.fecha, g.chla_media, marker="o", label=f"Lago de {lago}")
ax.axhline(UMBRAL_FLORACION, color="red", ls=":", label=f"Umbral {UMBRAL_FLORACION} mg/m3")
ax.set_title("Clorofila-a media: Atitlán vs Amatitlán")
ax.set_ylabel("Clorofila-a (mg/m3)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(DATOS / "resultados" / "comparacion_lagos.png", dpi=120)
plt.show()

### 6.2 Interpretación de los patrones temporales

**Amatitlán.** Se espera un nivel base alto y sostenido en todas las fechas: es un
lago pequeño, poco profundo y que recibe el río Villalobos con las aguas
residuales del área metropolitana. El exceso de nutrientes es permanente, así que
la floración no aparece y desaparece, fluctúa alrededor de un nivel ya elevado.

**Atitlán.** Se espera un nivel base bajo con episodios puntuales. El lago es
profundo (más de 300 m) y su volumen diluye los nutrientes, de modo que la
floración necesita un disparador: escorrentía tras las lluvias o mezcla de la
columna de agua.

**Estacionalidad.** Los valores más altos tienden a caer entre el final de la
época lluviosa y el inicio de la seca (noviembre–febrero) y en los meses cálidos
previos a las lluvias (marzo–mayo). La lluvia arrastra nutrientes de las cuencas
y, cuando cesa, el agua se estanca y se calienta: las dos condiciones que
favorecen a la cianobacteria.

**Factores asociados a los picos:** descargas de aguas residuales y agrícolas,
temperatura del agua, estabilidad de la columna (poco viento y poca mezcla) y el
tiempo transcurrido desde la última lluvia fuerte.
